In [ ]:
import sys
sys.path.insert(0, '../app/src')

import os
import glob
import random
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import Configuration
from src.data.filter import (
    local_normalize_image,
    get_integral_image,
    get_integral_squared_image,
    get_std_from_integral_images,
)

CONFIG = Configuration()


In [ ]:
def process_image(path: str):
    img = np.array(Image.open(path).convert('L'))
    integral = get_integral_image(img)
    integral_2 = get_integral_squared_image(img)
    H, W = img.shape
    half = CONFIG.normalize_window // 2

    rows = np.arange(H)
    cols = np.arange(W)
    r1 = np.maximum(0, rows - half)[:, None]
    r2 = np.minimum(H - 1, rows + half)[:, None]
    c1 = np.maximum(0, cols - half)[None, :]
    c2 = np.minimum(W - 1, cols + half)[None, :]

    _, std_img = get_std_from_integral_images(integral, integral_2, r1, r2, c1, c2)
    norm_img = local_normalize_image(CONFIG, img)
    return img, norm_img, std_img


# Crowd Images – Interactive Filter Viewer
Use the buttons below to switch between crowd images. Each view shows the original image, the locally normalized version, and the per-pixel standard deviation (computed via integral images).

In [ ]:
crowd_paths = [f'../data/ViolaJones/crowd{i}.png' for i in range(1, 6)]
crowd_data = [process_image(p) for p in crowd_paths]
crowd_sizes = [(img.shape[1], img.shape[0]) for img, _, _ in crowd_data]


In [ ]:
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Original', 'Local Normalized', 'Std Dev'),
    vertical_spacing=0.05,
)

for i, (img, norm, std) in enumerate(crowd_data):
    visible = i == 0
    fig.add_trace(
        go.Heatmap(z=img[::-1], colorscale='gray', showscale=False, visible=visible, hoverongaps=False),
        row=1, col=1,
    )
    fig.add_trace(
        go.Heatmap(z=norm[::-1], colorscale='gray', showscale=False, visible=visible, hoverongaps=False),
        row=2, col=1,
    )
    fig.add_trace(
        go.Heatmap(z=std[::-1], colorscale='Hot', showscale=False, visible=visible, hoverongaps=False),
        row=3, col=1,
    )

buttons = []
for i in range(len(crowd_data)):
    W, H = crowd_sizes[i]
    target_w = 800
    scale = target_w / W
    w = int(target_w + 150)
    h = int(3 * H * scale + 250)
    vis = [False] * (len(crowd_data) * 3)
    vis[i * 3] = True
    vis[i * 3 + 1] = True
    vis[i * 3 + 2] = True
    buttons.append(
        dict(
            label=f'Crowd {i + 1}',
            method='update',
            args=[{'visible': vis}, {'width': w, 'height': h}],
        )
    )

# Initialize with first image size
W0, H0 = crowd_sizes[0]
scale0 = 800 / W0
init_w = int(800 + 150)
init_h = int(3 * H0 * scale0 + 250)

fig.update_layout(
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=0.1,
            y=1.15,
            buttons=buttons,
        )
    ],
    title_text='Crowd Images with Filters',
    width=init_w,
    height=init_h,
    autosize=False,
)
fig.update_xaxes(constrain='domain')
fig.update_yaxes(scaleanchor='x', scaleratio=1, constrain='domain')
fig.show()


# Random Face Images – Filter Grid
A static grid of random samples from `CONFIG.faces_all_path` showing the same three filter views side-by-side.

In [ ]:
faces_pattern = os.path.join(CONFIG.faces_all_path, '*.jpg')
faces_paths = glob.glob(faces_pattern)
random.seed(CONFIG.seed)
sample_faces = random.sample(faces_paths, min(6, len(faces_paths)))
face_data = [process_image(p) for p in sample_faces]


In [ ]:
n_faces = len(face_data)
row_h = 300
col_w = 300
fig2 = make_subplots(
    rows=n_faces,
    cols=3,
    subplot_titles=[
        title
        for _ in range(n_faces)
        for title in ('Original', 'Local Normalized', 'Std Dev')
    ],
    vertical_spacing=0.05,
)

for i, (img, norm, std) in enumerate(face_data):
    row = i + 1
    fig2.add_trace(
        go.Heatmap(z=img[::-1], colorscale='gray', showscale=False, hoverongaps=False),
        row=row, col=1,
    )
    fig2.add_trace(
        go.Heatmap(z=norm[::-1], colorscale='gray', showscale=False, hoverongaps=False),
        row=row, col=2,
    )
    fig2.add_trace(
        go.Heatmap(z=std[::-1], colorscale='Hot', showscale=False, hoverongaps=False),
        row=row, col=3,
    )

fig2.update_layout(
    width=col_w * 3 + 200,
    height=row_h * n_faces + 100,
    title_text='Random Face Images with Filters',
    showlegend=False,
    autosize=False,
)
fig2.update_xaxes(constrain='domain')
fig2.update_yaxes(scaleanchor='x', scaleratio=1, constrain='domain')
fig2.show()
